In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescent(
    model=model,
    lr_init=1e-4,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.48047733306884766
epoch:  1, loss: 0.4802403151988983
epoch:  2, loss: 0.48000332713127136
epoch:  3, loss: 0.4797663986682892
epoch:  4, loss: 0.47952964901924133
epoch:  5, loss: 0.47929298877716064
epoch:  6, loss: 0.47905638813972473
epoch:  7, loss: 0.478819876909256
epoch:  8, loss: 0.4785834550857544
epoch:  9, loss: 0.47834721207618713
epoch:  10, loss: 0.4781109690666199
epoch:  11, loss: 0.47787487506866455
epoch:  12, loss: 0.4776388108730316
epoch:  13, loss: 0.47740286588668823
epoch:  14, loss: 0.4771670997142792
epoch:  15, loss: 0.4769313931465149
epoch:  16, loss: 0.4766957759857178
epoch:  17, loss: 0.47646021842956543
epoch:  18, loss: 0.476224809885025
epoch:  19, loss: 0.47598952054977417
epoch:  20, loss: 0.47575435042381287
epoch:  21, loss: 0.47551921010017395
epoch:  22, loss: 0.475284218788147
epoch:  23, loss: 0.47504937648773193
epoch:  24, loss: 0.4748145341873169
epoch:  25, loss: 0.474579781293869
epoch:  26, loss: 0.4743451774120331
ep

In [7]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7527934136505006
Test metrics:  R2 = 0.7403091392745913


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="scaled",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.13895300030708313
epoch:  1, loss: 0.09778451919555664
epoch:  2, loss: 0.05976675823330879
epoch:  3, loss: 0.03301869332790375
epoch:  4, loss: 0.03263159841299057
epoch:  5, loss: 0.03233465552330017
epoch:  6, loss: 0.03230540454387665
epoch:  7, loss: 0.032280564308166504
epoch:  8, loss: 0.032260093837976456
epoch:  9, loss: 0.032248202711343765
epoch:  10, loss: 0.032220274209976196
epoch:  11, loss: 0.03219955042004585
epoch:  12, loss: 0.03218947350978851
epoch:  13, loss: 0.032171741127967834
epoch:  14, loss: 0.03216780349612236
epoch:  15, loss: 0.032163865864276886
epoch:  16, loss: 0.03215958923101425
epoch:  17, loss: 0.03215523064136505
epoch:  18, loss: 0.03215094655752182
epoch:  19, loss: 0.03214674070477486
epoch:  20, loss: 0.032142557203769684
epoch:  21, loss: 0.032138433307409286
epoch:  22, loss: 0.03213435783982277
epoch:  23, loss: 0.03213031217455864
epoch:  24, loss: 0.032126326113939285
epoch:  25, loss: 0.0321224108338356
epoch:  26, lo

In [9]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -1273.9253129151643
Test metrics:  R2 = -1312.2093835530507


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="BB1",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.32931259274482727
epoch:  1, loss: 0.186345174908638
epoch:  2, loss: 0.11089836061000824
epoch:  3, loss: 0.0715814009308815
epoch:  4, loss: 0.051508769392967224
epoch:  5, loss: 0.041473690420389175
epoch:  6, loss: 0.03654887154698372
epoch:  7, loss: 0.03416713699698448
epoch:  8, loss: 0.03302663192152977
epoch:  9, loss: 0.032483331859111786
epoch:  10, loss: 0.03222420811653137
epoch:  11, loss: 0.03209923952817917
epoch:  12, loss: 0.032037220895290375
epoch:  13, loss: 0.032004520297050476
epoch:  14, loss: 0.031985606998205185
epoch:  15, loss: 0.031953100115060806
epoch:  16, loss: 0.031918954104185104
epoch:  17, loss: 0.03189926967024803
epoch:  18, loss: 0.03186669945716858
epoch:  19, loss: 0.031831830739974976
epoch:  20, loss: 0.03181188926100731
epoch:  21, loss: 0.03178462013602257
epoch:  22, loss: 0.03174867480993271
epoch:  23, loss: 0.03172861784696579
epoch:  24, loss: 0.03170739486813545
epoch:  25, loss: 0.03167048469185829
epoch:  26, loss

In [11]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7855715402380818
Test metrics:  R2 = 0.7601084118617009


In [12]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="BB2",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.32426953315734863
epoch:  1, loss: 0.19436904788017273
epoch:  2, loss: 0.12365718185901642
epoch:  3, loss: 0.08440925925970078
epoch:  4, loss: 0.06195053085684776
epoch:  5, loss: 0.04911509156227112
epoch:  6, loss: 0.041812099516391754
epoch:  7, loss: 0.03767772018909454
epoch:  8, loss: 0.035348013043403625
epoch:  9, loss: 0.03404022380709648
epoch:  10, loss: 0.033308014273643494
epoch:  11, loss: 0.032898545265197754
epoch:  12, loss: 0.0326693132519722
epoch:  13, loss: 0.03254052996635437
epoch:  14, loss: 0.03246770426630974
epoch:  15, loss: 0.032425954937934875
epoch:  16, loss: 0.03240130841732025
epoch:  17, loss: 0.032386213541030884
epoch:  18, loss: 0.03237532079219818
epoch:  19, loss: 0.03235667198896408
epoch:  20, loss: 0.032345131039619446
epoch:  21, loss: 0.032333388924598694
epoch:  22, loss: 0.03231878578662872
epoch:  23, loss: 0.03231450170278549
epoch:  24, loss: 0.03229597210884094
epoch:  25, loss: 0.03228475898504257
epoch:  26, los

In [13]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7520383703246627
Test metrics:  R2 = 0.7365969148822673


In [14]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="quadratic",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5886863470077515
epoch:  1, loss: 0.3410429060459137
epoch:  2, loss: 0.3410429060459137
epoch:  3, loss: 0.3410429060459137
epoch:  4, loss: 0.3410429060459137
epoch:  5, loss: 0.3410429060459137
epoch:  6, loss: 0.3410429060459137
epoch:  7, loss: 0.3410429060459137
epoch:  8, loss: 0.3410429060459137
epoch:  9, loss: 0.3410429060459137
epoch:  10, loss: 0.3410429060459137
epoch:  11, loss: 0.3410429060459137
epoch:  12, loss: 0.3410429060459137
epoch:  13, loss: 0.3410429060459137
epoch:  14, loss: 0.3410429060459137
epoch:  15, loss: 0.3410429060459137
epoch:  16, loss: 0.3410429060459137
epoch:  17, loss: 0.3410429060459137
epoch:  18, loss: 0.3410429060459137
epoch:  19, loss: 0.3410429060459137
epoch:  20, loss: 0.3410429060459137
epoch:  21, loss: 0.3410429060459137
epoch:  22, loss: 0.3410429060459137
epoch:  23, loss: 0.3410429060459137
epoch:  24, loss: 0.3410429060459137
epoch:  25, loss: 0.3410429060459137
epoch:  26, loss: 0.3410429060459137
epoch:  27,

In [15]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -85625.8219487475
Test metrics:  R2 = -79145.29827447237


In [16]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="lipschitz",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.08484649658203125
epoch:  1, loss: 0.0581401064991951
epoch:  2, loss: 0.045027703046798706
epoch:  3, loss: 0.03858074173331261
epoch:  4, loss: 0.03542983904480934
epoch:  5, loss: 0.03389771655201912
epoch:  6, loss: 0.03315158560872078
epoch:  7, loss: 0.03278432786464691
epoch:  8, loss: 0.032602228224277496
epoch:  9, loss: 0.032509077340364456
epoch:  10, loss: 0.03245820105075836
epoch:  11, loss: 0.032453909516334534
epoch:  12, loss: 0.032366376370191574
epoch:  13, loss: 0.032319508492946625
epoch:  14, loss: 0.03229247033596039
epoch:  15, loss: 0.032264236360788345
epoch:  16, loss: 0.03222179040312767
epoch:  17, loss: 0.03219682350754738
epoch:  18, loss: 0.03216966241598129
epoch:  19, loss: 0.032128021121025085
epoch:  20, loss: 0.03210420534014702
epoch:  21, loss: 0.03208531439304352
epoch:  22, loss: 0.03204623609781265
epoch:  23, loss: 0.03202347829937935
epoch:  24, loss: 0.03199911490082741
epoch:  25, loss: 0.03196105360984802
epoch:  26, los

In [17]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8157634281570103
Test metrics:  R2 = 0.8208279401209491


In [18]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="keep",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.7032565474510193
epoch:  1, loss: 0.3711807131767273
epoch:  2, loss: 0.20775917172431946
epoch:  3, loss: 0.12314599752426147
epoch:  4, loss: 0.07877803593873978
epoch:  5, loss: 0.05575608089566231
epoch:  6, loss: 0.04398419335484505
epoch:  7, loss: 0.03805684298276901
epoch:  8, loss: 0.03513118624687195
epoch:  9, loss: 0.033691249787807465
epoch:  10, loss: 0.032971933484077454
epoch:  11, loss: 0.03260960429906845
epoch:  12, loss: 0.032426729798316956
epoch:  13, loss: 0.03233398497104645
epoch:  14, loss: 0.03228644281625748
epoch:  15, loss: 0.03226137533783913
epoch:  16, loss: 0.03224724903702736
epoch:  17, loss: 0.032238759100437164
epoch:  18, loss: 0.032233402132987976
epoch:  19, loss: 0.032229434698820114
epoch:  20, loss: 0.03222616761922836
epoch:  21, loss: 0.0322231650352478
epoch:  22, loss: 0.032220181077718735
epoch:  23, loss: 0.032217252999544144
epoch:  24, loss: 0.03221423551440239
epoch:  25, loss: 0.03221132978796959
epoch:  26, loss:

In [19]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.17638128687554544
Test metrics:  R2 = 0.13237645482486138
